# Practical 3 - Mapping, Text vs Keyword & Advanced Field Types

This notebook uses the official **elasticsearch** Python client against a local
Elasticsearch 9.x node (`http://127.0.0.1:9200`).

Tasks:
1. Explicit mapping for the `products` index.
2. Dynamic mapping on an index created without mappings.
3. Text vs keyword behaviour (match / term / sort / aggregations).
4. Advanced field types: object, nested, flattened, date, geo_point, multi-fields.
5. Mapping conflicts and the errors they produce.

Every experiment is reproducible: indices are dropped and recreated at the start
of the relevant section, and conflict tests are wrapped in `try/except` so the
notebook always runs top-to-bottom.

In [1]:
from elasticsearch import Elasticsearch, RequestError
import json

ES_URL = "http://127.0.0.1:9200"
es = Elasticsearch(ES_URL)

info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])


def pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def drop(index):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
        print("deleted existing index:", index)


def recreate(index, mappings=None):
    drop(index)
    kwargs = {"settings": {"number_of_shards": 1, "number_of_replicas": 0}}
    if mappings is not None:
        kwargs["mappings"] = mappings
    es.indices.create(index=index, **kwargs)
    print("created index:", index)


def show_mapping(index):
    print(pretty(dict(es.indices.get_mapping(index=index))))


def hit_count(response):
    return response["hits"]["total"]["value"]


def describe_error(error):
    err = getattr(error, "error", None)
    if isinstance(err, dict):
        return err.get("type"), err.get("reason")
    body = getattr(error, "body", None)
    if isinstance(body, dict) and isinstance(body.get("error"), dict):
        return body["error"].get("type"), body["error"].get("reason")
    reason = getattr(error, "info", None)
    if isinstance(reason, dict):
        inner = reason.get("error", {})
        if isinstance(inner, dict):
            return inner.get("type", err), inner.get("reason", str(error))
    return (err or "error"), str(error)

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node


## Task 1 - Explicit Mapping

We define every field up front. `number_of_shards: 1` and
`number_of_replicas: 0` keep the index **green** on a single-node cluster.

Note the type choices:
- `product_id`, `category` -> `keyword` (exact values, aggregatable/sortable)
- `product_name`, `description` -> `text` (full-text search)
- `price` -> `float`, `quantity` -> `integer`
- `available` -> `boolean`, `created_at` -> `date`
- `location` -> `geo_point` (lat/lon)

In [2]:
products_mapping = {
    "properties": {
        "product_id":   {"type": "keyword"},
        "product_name": {"type": "text"},
        "category":     {"type": "keyword"},
        "description":  {"type": "text"},
        "price":        {"type": "float"},
        "quantity":     {"type": "integer"},
        "available":    {"type": "boolean"},
        "created_at":   {"type": "date"},
        "location":     {"type": "geo_point"},
    }
}

recreate("products", products_mapping)
show_mapping("products")

deleted existing index: products


created index: products
{
  "products": {
    "mappings": {
      "properties": {
        "available": {
          "type": "boolean"
        },
        "category": {
          "type": "keyword"
        },
        "created_at": {
          "type": "date"
        },
        "description": {
          "type": "text"
        },
        "location": {
          "type": "geo_point"
        },
        "price": {
          "type": "float"
        },
        "product_id": {
          "type": "keyword"
        },
        "product_name": {
          "type": "text"
        },
        "quantity": {
          "type": "integer"
        }
      }
    }
  }
}


In [3]:
products = [
    {
        "product_id": "P001",
        "product_name": "Green Valley Residency",
        "category": "real_estate",
        "description": "A quiet residential community surrounded by greenery.",
        "price": 7500000.0,
        "quantity": 5,
        "available": True,
        "created_at": "2024-01-15",
        "location": {"lat": 19.0760, "lon": 72.8777},
    },
    {
        "product_id": "P002",
        "product_name": "Wireless Noise-Cancelling Headphones",
        "category": "electronics",
        "description": "Over-ear headphones with active noise cancellation.",
        "price": 299.99,
        "quantity": 40,
        "available": True,
        "created_at": "2024-03-02",
        "location": {"lat": 28.6139, "lon": 77.2090},
    },
    {
        "product_id": "P003",
        "product_name": "Ceramic Coffee Mug",
        "category": "home_kitchen",
        "description": "350 ml ceramic mug, dishwasher safe.",
        "price": 9.5,
        "quantity": 0,
        "available": False,
        "created_at": "2024-05-20",
        "location": {"lat": 12.9716, "lon": 77.5946},
    },
]

for doc in products:
    es.index(index="products", id=doc["product_id"], document=doc)

es.indices.refresh(index="products")
print("documents in products:", es.count(index="products")["count"])

documents in products: 3


## Task 2 - Dynamic Mapping

We create `products_dynamic` **without any mappings**. Elasticsearch then infers
field types from the first document it sees. Observe:

- strings -> `text` + a `.keyword` sub-field
- integers -> `long`, decimals -> `float`
- true/false -> `boolean`
- ISO date strings -> `date`
- JSON objects -> `object`
- arrays -> handled element-wise

New fields added later are also inferred automatically (dynamic mapping stays on).

In [4]:
recreate("products_dynamic")  # no mappings argument at all

dynamic_docs = [
    {"name": "Laptop", "price": 1299.99, "stock": 25, "in_stock": True,
     "added": "2024-03-01", "tags": ["electronics", "computers"]},
    {"name": "Coffee Mug", "price": 9, "stock": 100, "in_stock": True,
     "added": "2024-04-10", "specs": {"material": "ceramic", "capacity_ml": 350}},
    {"name": "Notebook", "price": 4.5, "stock": 0, "in_stock": False,
     "added": "2024-05-20"},
]

for i, doc in enumerate(dynamic_docs):
    es.index(index="products_dynamic", id=f"doc-{i + 1}", document=doc)

es.indices.refresh(index="products_dynamic")
print("Mapping inferred after the first inserts:")
show_mapping("products_dynamic")

es.index(index="products_dynamic", id="doc-4", document={
    "name": "Desk Lamp", "price": 45.0, "stock": 12, "in_stock": True,
    "added": "2024-06-01", "warranty_years": 2,
    "supplier": {"name": "LuxCo", "country": "IN"},
})
es.indices.refresh(index="products_dynamic")
print("\nMapping after adding new fields (warranty_years, supplier):")
show_mapping("products_dynamic")

deleted existing index: products_dynamic


created index: products_dynamic


Mapping inferred after the first inserts:
{
  "products_dynamic": {
    "mappings": {
      "properties": {
        "added": {
          "type": "date"
        },
        "in_stock": {
          "type": "boolean"
        },
        "name": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "price": {
          "type": "float"
        },
        "specs": {
          "properties": {
            "capacity_ml": {
              "type": "long"
            },
            "material": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            }
          }
        },
        "stock": {
          "type": "long"
        },
        "tags": {
          "type": "text",
          "fields": {
            "keyword": {
    

{
  "products_dynamic": {
    "mappings": {
      "properties": {
        "added": {
          "type": "date"
        },
        "in_stock": {
          "type": "boolean"
        },
        "name": {
          "type": "text",
          "fields": {
            "keyword": {
              "type": "keyword",
              "ignore_above": 256
            }
          }
        },
        "price": {
          "type": "float"
        },
        "specs": {
          "properties": {
            "capacity_ml": {
              "type": "long"
            },
            "material": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            }
          }
        },
        "stock": {
          "type": "long"
        },
        "supplier": {
          "properties": {
            "country": {
              "type": "text",
              "fields": {
        

## Task 3 - Text vs Keyword

We insert `Green Valley Residency` into a field that has both representations:

- `product_name` is `text` -> analyzed at index time into tokens
  `[green, valley, residency]` (lowercased).
- `product_name.keyword` is `keyword` -> stored as one exact string
  `Green Valley Residency`.

This single multi-field lets us compare both behaviours on identical data.

In [5]:
text_keyword_mapping = {
    "properties": {
        "product_name": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
        },
        "tags": {"type": "keyword"},
    }
}
recreate("text_vs_keyword", text_keyword_mapping)

es.index(index="text_vs_keyword", id="1", document={
    "product_name": "Green Valley Residency",
    "tags": ["residency", "green"],
})
es.indices.refresh(index="text_vs_keyword")
print(es.get(index="text_vs_keyword", id="1")["_source"])

deleted existing index: text_vs_keyword


created index: text_vs_keyword
{'product_name': 'Green Valley Residency', 'tags': ['residency', 'green']}


In [6]:
res = es.search(index="text_vs_keyword", query={"match": {"product_name": "green valley"}})
print("match on TEXT  ('green valley')      -> hits:", hit_count(res))

res = es.search(index="text_vs_keyword", query={"match": {"product_name": "Green Valley Residency"}})
print("match on TEXT  (full phrase)         -> hits:", hit_count(res))

match on TEXT  ('green valley')      -> hits: 1


match on TEXT  (full phrase)         -> hits: 1


In [7]:
for value in ["green", "Green", "Green Valley Residency", "green valley residency"]:
    res = es.search(index="text_vs_keyword", query={"term": {"product_name": value}})
    print(f"term on TEXT    {value!r:30} -> hits: {hit_count(res)}")

term on TEXT    'green'                        -> hits: 1
term on TEXT    'Green'                        -> hits: 0


term on TEXT    'Green Valley Residency'       -> hits: 0


term on TEXT    'green valley residency'       -> hits: 0


In [8]:
for value in ["Green Valley Residency", "green valley residency", "Green"]:
    res = es.search(index="text_vs_keyword", query={"term": {"product_name.keyword": value}})
    print(f"term on KEYWORD {value!r:30} -> hits: {hit_count(res)}")

term on KEYWORD 'Green Valley Residency'       -> hits: 1
term on KEYWORD 'green valley residency'       -> hits: 0


term on KEYWORD 'Green'                        -> hits: 0


In [9]:
try:
    res = es.search(index="text_vs_keyword", sort=[{"product_name": "asc"}])
    print("sort on TEXT    -> OK")
except RequestError as error:
    print("sort on TEXT    -> FAILED")
    print(pretty(error.error))

res = es.search(index="text_vs_keyword", sort=[{"product_name.keyword": "asc"}])
print("sort on KEYWORD -> OK")

sort on TEXT    -> FAILED
"search_phase_execution_exception"


sort on KEYWORD -> OK


In [10]:
try:
    res = es.search(index="text_vs_keyword", size=0,
                    aggs={"names": {"terms": {"field": "product_name"}}})
    print("aggregation on TEXT    -> OK")
except RequestError as error:
    print("aggregation on TEXT    -> FAILED")
    print(pretty(error.error))

res = es.search(index="text_vs_keyword", size=0,
                aggs={"names": {"terms": {"field": "product_name.keyword"}}})
print("aggregation on KEYWORD -> OK")
print(pretty(res["aggregations"]))

aggregation on TEXT    -> FAILED
"search_phase_execution_exception"


aggregation on KEYWORD -> OK
{
  "names": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "Green Valley Residency",
        "doc_count": 1
      }
    ]
  }
}


### Why they differ

- **`text` is analyzed.** At index time `Green Valley Residency` becomes the
  tokens `green`, `valley`, `residency`. At query time `match` analyzes the query
  the same way, so `"green valley"` matches. `term` does **not** analyze: it looks
  for the literal token, so `"Green Valley Residency"` (with spaces/caps) matches
  nothing, while `"green"` matches because it is exactly one of the indexed tokens.
- **`keyword` is not analyzed.** The whole string is stored verbatim, so `term`
  matches the exact case-sensitive value `Green Valley Residency`, and fails on the
  lowercase variant.
- **Sorting and aggregations need doc values.** `text` fields have `fielddata`
  disabled by default (it is expensive), so sorting/aggregating on `product_name`
  raises `illegal_argument_exception`. The `keyword` multi-field works fine.

Rule of thumb: use `text` for full-text search and `keyword` (or `.keyword`) for
exact matching, sorting, and aggregations.

## Task 4 - Advanced Field Types

We build `advanced_types` with:

- `manufacturer` -> **object** (single nested object)
- `reviews` -> **nested** (array of objects that must stay correlated)
- `reviews_flat` -> **object** (same data, but object arrays are flattened)
- `attributes` -> **flattened** (arbitrary key/value pairs, stored as keywords)
- `created_at` -> **date**
- `location` -> **geo_point**
- `title` -> **text** with a `keyword` **multi-field**

In [11]:
advanced_mapping = {
    "properties": {
        "product_id": {"type": "keyword"},
        "manufacturer": {
            "properties": {
                "name": {"type": "keyword"},
                "country": {"type": "keyword"},
            }
        },
        "reviews": {
            "type": "nested",
            "properties": {
                "author": {"type": "keyword"},
                "rating": {"type": "integer"},
                "comment": {"type": "text"},
            },
        },
        "reviews_flat": {
            "properties": {
                "author": {"type": "keyword"},
                "rating": {"type": "integer"},
                "comment": {"type": "text"},
            }
        },
        "attributes": {"type": "flattened"},
        "created_at": {"type": "date"},
        "location": {"type": "geo_point"},
        "title": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
        },
    }
}
recreate("advanced_types", advanced_mapping)
show_mapping("advanced_types")

deleted existing index: advanced_types


created index: advanced_types
{
  "advanced_types": {
    "mappings": {
      "properties": {
        "attributes": {
          "type": "flattened"
        },
        "created_at": {
          "type": "date"
        },
        "location": {
          "type": "geo_point"
        },
        "manufacturer": {
          "properties": {
            "country": {
              "type": "keyword"
            },
            "name": {
              "type": "keyword"
            }
          }
        },
        "product_id": {
          "type": "keyword"
        },
        "reviews": {
          "type": "nested",
          "properties": {
            "author": {
              "type": "keyword"
            },
            "comment": {
              "type": "text"
            },
            "rating": {
              "type": "integer"
            }
          }
        },
        "reviews_flat": {
          "properties": {
            "author": {
              "type": "keyword"
            },
         

In [12]:
review_set_a = [
    {"author": "alice", "rating": 5, "comment": "excellent quality"},
    {"author": "bob", "rating": 3, "comment": "average"},
]
review_set_b = [
    {"author": "alice", "rating": 2, "comment": "not great"},
    {"author": "bob", "rating": 5, "comment": "loved it"},
]

advanced_docs = [
    {
        "product_id": "A1",
        "title": "Premium Widget",
        "manufacturer": {"name": "Acme", "country": "US"},
        "reviews": review_set_a,
        "reviews_flat": review_set_a,
        "attributes": {"color": "red", "size": "L", "warranty": "2 years"},
        "created_at": "2024-02-01",
        "location": {"lat": 28.6139, "lon": 77.2090},
    },
    {
        "product_id": "A2",
        "title": "Budget Widget",
        "manufacturer": {"name": "Globex", "country": "IN"},
        "reviews": review_set_b,
        "reviews_flat": review_set_b,
        "attributes": {"color": "blue", "size": "M", "warranty": "1 year"},
        "created_at": "2024-08-15",
        "location": {"lat": 19.0760, "lon": 72.8777},
    },
]
for doc in advanced_docs:
    es.index(index="advanced_types", id=doc["product_id"], document=doc)
es.indices.refresh(index="advanced_types")
print("indexed documents:", es.count(index="advanced_types")["count"])

indexed documents: 2


In [13]:
object_query = {
    "bool": {
        "must": [
            {"term": {"reviews_flat.author": "alice"}},
            {"range": {"reviews_flat.rating": {"gte": 4}}},
        ]
    }
}
nested_query = {
    "nested": {
        "path": "reviews",
        "query": {
            "bool": {
                "must": [
                    {"term": {"reviews.author": "alice"}},
                    {"range": {"reviews.rating": {"gte": 4}}},
                ]
            }
        },
    }
}

object_hits = es.search(index="advanced_types", query=object_query)["hits"]["hits"]
nested_hits = es.search(index="advanced_types", query=nested_query)["hits"]["hits"]
print("OBJECT query (author=alice AND rating>=4):", [h["_source"]["product_id"] for h in object_hits])
print("NESTED query (author=alice AND rating>=4):", [h["_source"]["product_id"] for h in nested_hits])
print()
print("A2 has alice rating=2 and bob rating=5 (no single review satisfies both).")
print("The OBJECT query wrongly matches A2 because object arrays are flattened.")
print("The NESTED query correctly does NOT match A2 because each review is independent.")

OBJECT query (author=alice AND rating>=4): ['A1', 'A2']
NESTED query (author=alice AND rating>=4): ['A1']

A2 has alice rating=2 and bob rating=5 (no single review satisfies both).
The OBJECT query wrongly matches A2 because object arrays are flattened.
The NESTED query correctly does NOT match A2 because each review is independent.


In [14]:
res = es.search(index="advanced_types", query={"term": {"attributes.color": "red"}})
print("flattened term attributes.color='red'      -> hits:", hit_count(res))

res = es.search(index="advanced_types", query={"term": {"attributes.warranty": "2 years"}})
print("flattened term attributes.warranty='2 years' -> hits:", hit_count(res))
print("Flattened keys/values are keyword-like: exact match, no text analysis.")

flattened term attributes.color='red'      -> hits:

 1
flattened term attributes.warranty='2 years' -> hits: 1
Flattened keys/values are keyword-like: exact match, no text analysis.


In [15]:
res = es.search(index="advanced_types",
                query={"range": {"created_at": {"gte": "2024-06-01", "lte": "2024-12-31"}}})
print("date range (2024-06-01..2024-12-31) hits:", hit_count(res))

res = es.search(index="advanced_types",
                query={"geo_distance": {"distance": "100km",
                                        "location": {"lat": 19.0760, "lon": 72.8777}}})
print("geo_distance (100km from Mumbai) hits:", hit_count(res))
print("inside:", [h["_source"]["product_id"] for h in res["hits"]["hits"]])

date range (2024-06-01..2024-12-31) hits: 1


geo_distance (100km from Mumbai) hits: 1
inside: ['A2']


In [16]:
res = es.search(index="advanced_types", query={"match": {"title": "premium widget"}})
print("match on title (text)            hits:", hit_count(res))

res = es.search(index="advanced_types", query={"term": {"title.keyword": "Premium Widget"}})
print("term on title.keyword (exact)    hits:", hit_count(res))

res = es.search(index="advanced_types", size=0,
                aggs={"titles": {"terms": {"field": "title.keyword"}}})
print("terms aggregation on title.keyword:")
print(pretty(res["aggregations"]))

match on title (text)            hits: 2


term on title.keyword (exact)    hits: 1


terms aggregation on title.keyword:
{
  "titles": {
    "doc_count_error_upper_bound": 0,
    "sum_other_doc_count": 0,
    "buckets": [
      {
        "key": "Budget Widget",
        "doc_count": 1
      },
      {
        "key": "Premium Widget",
        "doc_count": 1
      }
    ]
  }
}


### What this shows

- **object**: convenient for a single embedded object (`manufacturer`), but arrays
  of objects are flattened, so you cannot correlate fields across array elements.
- **nested**: each element is indexed as a hidden separate document, so
  cross-field conditions (author AND rating in the *same* review) work correctly.
  Query nested fields with a `nested` query.
- **flattened**: a bag of arbitrary keys mapped to keyword-like values. Great for
  unpredictable attributes, but no text analysis and limited query types.
- **date**: supports `range` queries and date math.
- **geo_point**: supports `geo_distance`, bounding box, and geo aggregations.
- **multi-fields**: one source value exposed as both `text` (search) and
  `keyword` (sort/aggregate) via `title` and `title.keyword`.

## Task 5 - Mapping Conflicts

Mappings are **immutable for existing fields**: you cannot change a field's type
after it is created. Writing incompatible data raises a parse error and the
document is rejected (the mapping stays unchanged).

Below, expected failures are caught so the notebook keeps running.

In [17]:
def attempt(index, doc_id, document):
    try:
        response = es.index(index=index, id=doc_id, document=document)
        es.indices.refresh(index=index)
        print(f"ACCEPTED  id={doc_id:16} result={response['result']}")
    except RequestError as error:
        err_type, err_reason = describe_error(error)
        print(f"REJECTED  id={doc_id:16} type={err_type}")
        print("          reason:", err_reason)


base_product = {
    "product_id": "PX",
    "product_name": "Conflict Test",
    "category": "test",
    "description": "mapping conflict demonstration",
    "price": 100.0,
    "quantity": 1,
    "available": True,
    "created_at": "2024-01-01",
    "location": {"lat": 0.0, "lon": 0.0},
}

print("-- price (float) receives the string 'expensive' --")
attempt("products", "conflict-price", {**base_product, "price": "expensive"})

print("\n-- quantity (integer) receives the string 'ten' --")
attempt("products", "conflict-quantity", {**base_product, "quantity": "ten"})

print("\n-- available (boolean) receives 'maybe' --")
attempt("products", "conflict-available", {**base_product, "available": "maybe"})

print("\n-- created_at (date) receives 'not-a-date' --")
attempt("products", "conflict-date", {**base_product, "created_at": "not-a-date"})

-- price (float) receives the string 'expensive' --


REJECTED  id=conflict-price   type=document_parsing_exception
          reason: [1:124] failed to parse field [price] of type [float] in document with id 'conflict-price'. Preview of field's value: 'expensive'

-- quantity (integer) receives the string 'ten' --


REJECTED  id=conflict-quantity type=document_parsing_exception


          reason: [1:141] failed to parse field [quantity] of type [integer] in document with id 'conflict-quantity'. Preview of field's value: 'ten'

-- available (boolean) receives 'maybe' --


REJECTED  id=conflict-available type=document_parsing_exception


          reason: [1:155] failed to parse field [available] of type [boolean] in document with id 'conflict-available'. Preview of field's value: 'maybe'

-- created_at (date) receives 'not-a-date' --
REJECTED  id=conflict-date    type=document_parsing_exception
          reason: [1:173] failed to parse field [created_at] of type [date] in document with id 'conflict-date'. Preview of field's value: 'not-a-date'


In [18]:
recreate("conflict_dynamic")

print("-- dynamic field 'rating' is first seen as a number (long) --")
attempt("conflict_dynamic", "c1", {"rating": 5})
attempt("conflict_dynamic", "c2", {"rating": 3})

print("\n-- now 'rating' receives a non-numeric string --")
attempt("conflict_dynamic", "c3", {"rating": "five"})

print("\n-- dynamic field 'meta' is first seen as an object --")
attempt("conflict_dynamic", "c4", {"meta": {"color": "red"}})

print("\n-- now 'meta' receives a scalar string --")
attempt("conflict_dynamic", "c5", {"meta": "just text"})

deleted existing index: conflict_dynamic


created index: conflict_dynamic
-- dynamic field 'rating' is first seen as a number (long) --


ACCEPTED  id=c1               result=created
ACCEPTED  id=c2               result=created

-- now 'rating' receives a non-numeric string --


REJECTED  id=c3               type=document_parsing_exception
          reason: [1:11] failed to parse field [rating] of type [long] in document with id 'c3'. Preview of field's value: 'five'

-- dynamic field 'meta' is first seen as an object --


ACCEPTED  id=c4               result=created

-- now 'meta' receives a scalar string --


REJECTED  id=c5               type=document_parsing_exception
          reason: [1:9] object mapping for [meta] tried to parse field [meta] as object, but found a concrete value


In [19]:
print("products mapping is unchanged after the rejected writes:")
show_mapping("products")

print("\nconflict_dynamic mapping (only fields that were inferred successfully):")
show_mapping("conflict_dynamic")

products mapping is unchanged after the rejected writes:
{
  "products": {
    "mappings": {
      "properties": {
        "available": {
          "type": "boolean"
        },
        "category": {
          "type": "keyword"
        },
        "created_at": {
          "type": "date"
        },
        "description": {
          "type": "text"
        },
        "location": {
          "type": "geo_point"
        },
        "price": {
          "type": "float"
        },
        "product_id": {
          "type": "keyword"
        },
        "product_name": {
          "type": "text"
        },
        "quantity": {
          "type": "integer"
        }
      }
    }
  }
}

conflict_dynamic mapping (only fields that were inferred successfully):
{
  "conflict_dynamic": {
    "mappings": {
      "properties": {
        "meta": {
          "properties": {
            "color": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": 

### Why conflicts happen and how to fix them

- Elasticsearch maps a field to a type on first use (or from an explicit mapping).
  After that the type is fixed.
- A value that cannot be parsed into that type (e.g. `"expensive"` into `float`,
  `"five"` into a dynamically-created `long`, a scalar into an `object`) raises a
  `document_parsing_exception` / `mapper_parsing_exception`. The document is not
  indexed.
- You cannot "change" a field's type in place. To fix it you must:
  1. create a **new index** with the correct mapping,
  2. **reindex** data into it (`POST _reindex`),
  3. switch reads/writes to the new index (optionally via an alias).
- Design mappings carefully up front, or use types that tolerate variation such
  as `flattened` for free-form attributes.

In [20]:
names = ["products", "products_dynamic", "text_vs_keyword",
         "advanced_types", "conflict_dynamic"]
existing = sorted(dict(es.indices.get(index=",".join(names))).keys())
print("Indices created in this practical:")
for name in existing:
    print(f"  {name:20} docs = {es.count(index=name)['count']}")

# Optional cleanup - uncomment to remove the demo indices:
# for name in ["products", "products_dynamic", "text_vs_keyword",
#              "advanced_types", "conflict_dynamic"]:
#     drop(name)

Indices created in this practical:
  advanced_types       docs = 2


  conflict_dynamic     docs = 3
  products             docs = 3
  products_dynamic     docs = 4
  text_vs_keyword      docs = 1
